# Aiyagari Deep Learning Solver with Endogenous Prices and RSVP Fields

This notebook implements a **toy Aiyagari-style heterogeneous agent model** with:

- **Endogenous prices**: interest rate \(r\) and wage \(w\) are pinned down by a Cobb–Douglas production function and the aggregate capital stock implied by the household distribution.
- A **neural network policy** \( a' = f_\theta(a,z) \) trained from the **Euler equation residual**.
- A simple **GE fixed-point iteration** on aggregate capital and prices.
- An **RSVP-style field view**:
  - Policy field \( \mathbf{v}(a,z) = (c(a,z), a'(a,z)) \)
  - Entropy field \( S(a,z) \) from the stationary cross-sectional distribution
  - Approximate value potential \( \Phi(a,z) \) via Monte Carlo rollout and a separate value network.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

np.random.seed(0)
torch.manual_seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


In [ ]:

# === Preference parameters ===
beta = 0.96
sigma = 2.0   # CRRA

# === Production parameters ===
alpha = 0.36
delta = 0.08
L = 1.0       # normalized labor supply

# === Asset bounds ===
a_min = 0.0
a_max = 50.0

# === Idiosyncratic income states ===
z_states = np.array([0.5, 1.5], dtype=np.float32)
P = np.array([[0.9, 0.1],
              [0.1, 0.9]], dtype=np.float32)

def prod_prices(K):
    \"\"\"Given aggregate capital K, return (r, w).\"\"\"
    K = max(K, 1e-4)
    Y = K**alpha * L**(1 - alpha)
    r = alpha * (K**(alpha - 1)) * L**(1 - alpha) - delta
    w = (1 - alpha) * (K**alpha) * (L**(-alpha))
    return r, w

def sample_next_z(z_idx):
    z_idx = np.asarray(z_idx, dtype=int)
    probs = P[z_idx]
    draws = np.random.rand(z_idx.shape[0])
    next_idx = (draws > probs[:, 0]).astype(int)
    return next_idx

# Initial guess for K and prices
K_guess = 5.0
r, w = prod_prices(K_guess)
r, w


In [ ]:

class PolicyNet(nn.Module):
    \"\"\"Policy network: (a, z) -> next-period assets a_next\"\"\"
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )
    def forward(self, a, z):
        x = torch.cat([a, z], dim=1)
        a_next = self.net(x)
        a_next = torch.nn.functional.softplus(a_next)
        return torch.clamp(a_next, a_min, a_max)

policy_net = PolicyNet().to(device)
policy_net


In [ ]:

def marginal_utility(c):
    eps = 1e-8
    return torch.clamp(c, eps).pow(-sigma)

def euler_residual(a, z, r, w, policy_net):
    \"\"\"Euler residual u'(c) - beta (1+r) E[u'(c_next)]\"\"\"
    # Current income and consumption
    y = w * z + (1 + r) * a
    a_next = policy_net(a, z)
    c = y - a_next
    mu_c = marginal_utility(c)
    
    # Next period shocks
    z_idx = (z.detach().cpu().numpy().reshape(-1) > 1.0).astype(int)
    z_next_idx = sample_next_z(z_idx)
    z_next_vals = z_states[z_next_idx]
    
    a_next_detached = a_next.detach().cpu().numpy().reshape(-1)
    a_next_t = torch.tensor(a_next_detached, dtype=torch.float32, device=device).view(-1, 1)
    z_next_t = torch.tensor(z_next_vals, dtype=torch.float32, device=device).view(-1, 1)
    
    y_next = w * z_next_t + (1 + r) * a_next_t
    a_next2 = policy_net(a_next_t, z_next_t)
    c_next = y_next - a_next2
    mu_c_next = marginal_utility(c_next)
    
    rhs = beta * (1 + r) * mu_c_next
    residual = mu_c - rhs
    return residual, c, c_next

def loss_batch(policy_net, r, w, batch_size=4096):
    a = torch.rand(batch_size, 1, device=device) * a_max
    z_idx = np.random.randint(0, 2, size=batch_size)
    z_vals = z_states[z_idx]
    z = torch.tensor(z_vals, dtype=torch.float32, device=device).view(-1, 1)
    
    residual, c, c_next = euler_residual(a, z, r, w, policy_net)
    loss_euler = (residual ** 2).mean()
    
    penalty_c = (torch.relu(-c + 1e-3) ** 2).mean()
    penalty_c_next = (torch.relu(-c_next + 1e-3) ** 2).mean()
    
    loss = loss_euler + 10.0 * (penalty_c + penalty_c_next)
    return loss, float(loss_euler.detach().cpu().item())


In [ ]:

def train_policy(policy_net, r, w, epochs=50, lr=1e-3, verbose=True):
    optimizer = optim.Adam(policy_net.parameters(), lr=lr)
    loss_hist, euler_hist = [], []
    for epoch in range(1, epochs+1):
        optimizer.zero_grad()
        loss, euler_mse = loss_batch(policy_net, r, w, batch_size=4096)
        loss.backward()
        optimizer.step()
        loss_hist.append(loss.item())
        euler_hist.append(euler_mse)
        if verbose and (epoch % 10 == 0 or epoch == 1):
            print(f"[train_policy] epoch {epoch:3d} | loss={loss.item():.5f} | Euler MSE={euler_mse:.5f}")
    return loss_hist, euler_hist


In [ ]:

def simulate_stationary(policy_net, r, w, T_sim=300, N_agents=5000, burn_in=100):
    a_agents = np.zeros((N_agents,), dtype=np.float32)
    z_idx_agents = np.zeros((N_agents,), dtype=int)
    
    K_path = []
    samples_a, samples_z = [], []
    
    for t in range(T_sim):
        a_t = torch.tensor(a_agents, dtype=torch.float32, device=device).view(-1, 1)
        z_vals = z_states[z_idx_agents]
        z_t = torch.tensor(z_vals, dtype=torch.float32, device=device).view(-1, 1)
        
        with torch.no_grad():
            y_t = w * z_t + (1 + r) * a_t
            a_next = policy_net(a_t, z_t)
            c_t = y_t - a_next
        
        a_agents = a_next.cpu().numpy().reshape(-1)
        a_agents = np.clip(a_agents, a_min, a_max)
        z_idx_agents = sample_next_z(z_idx_agents)
        
        K_path.append(a_agents.mean())
        
        if t >= burn_in:
            # collect some samples for RSVP fields
            samples_a.append(a_agents.copy())
            samples_z.append(z_idx_agents.copy())
    
    K_hat = np.mean(K_path[burn_in:])
    samples_a = np.concatenate(samples_a)
    samples_z = np.concatenate(samples_z)
    return K_hat, np.array(K_path), samples_a, samples_z


In [ ]:

# General equilibrium fixed-point loop (very simple and damped)

outer_iters = 5
K = K_guess
K_history = []
r_history = []
w_history = []

for it in range(outer_iters):
    r, w = prod_prices(K)
    print(f"\n[GE iter {it+1}] K={K:.3f}, r={r:.4f}, w={w:.4f}")
    
    # Train policy for current prices
    _ = train_policy(policy_net, r, w, epochs=50, lr=1e-3, verbose=True)
    
    # Simulate to get implied K
    K_hat, K_path, samples_a, samples_z = simulate_stationary(policy_net, r, w)
    print(f"[GE iter {it+1}] implied K_hat={K_hat:.3f}")
    
    # Damped update
    lam = 0.5
    K = lam * K + (1 - lam) * K_hat
    
    K_history.append(K)
    r_history.append(r)
    w_history.append(w)

print("\nFinal K guess:", K)
r, w = prod_prices(K)
r, w


In [ ]:

plt.figure(figsize=(6,4))
plt.plot(K_history, marker='o')
plt.xlabel("GE iteration")
plt.ylabel("Capital K")
plt.title("Convergence of aggregate capital")
plt.grid(True)
plt.show()

plt.figure(figsize=(6,4))
plt.plot(r_history, label='r')
plt.plot(w_history, label='w')
plt.xlabel("GE iteration")
plt.ylabel("Price")
plt.title("Convergence of prices")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:

# Final stationary simulation with converged K, r, w
r_eq, w_eq = prod_prices(K)
K_hat_final, K_path_final, samples_a, samples_z = simulate_stationary(
    policy_net, r_eq, w_eq, T_sim=400, N_agents=8000, burn_in=150
)

print("Final simulated K (check):", K_hat_final)


In [ ]:

# === Policy field v(a,z) on a grid ===
a_grid = np.linspace(a_min, a_max, 100, dtype=np.float32)
a_t = torch.tensor(a_grid, dtype=torch.float32, device=device).view(-1, 1)

policy_c = {}
policy_a_next = {}

with torch.no_grad():
    for z_val in z_states:
        z_t = torch.full_like(a_t, z_val)
        y_t = w_eq * z_t + (1 + r_eq) * a_t
        a_next_t = policy_net(a_t, z_t)
        c_t = y_t - a_next_t
        policy_c[z_val] = c_t.cpu().numpy().reshape(-1)
        policy_a_next[z_val] = a_next_t.cpu().numpy().reshape(-1)

plt.figure(figsize=(6,4))
for z_val in z_states:
    plt.plot(a_grid, policy_a_next[z_val], label=f"z={z_val:.1f}")
plt.xlabel("a")
plt.ylabel("a'")
plt.title("Policy field component a'(a,z)")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(6,4))
for z_val in z_states:
    plt.plot(a_grid, policy_c[z_val], label=f"z={z_val:.1f}")
plt.xlabel("a")
plt.ylabel("c(a,z)")
plt.title("Policy field component c(a,z)")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:

# Build histogram over (a,z) and define S(a,z) ~ -log density

bins_a = 50
hist_low, hist_high = a_min, a_max
hist, edges = np.histogram(samples_a, bins=bins_a, range=(hist_low, hist_high))
bin_centers = 0.5 * (edges[:-1] + edges[1:])

# Overall density
density = hist / hist.sum()
eps = 1e-10
S_overall = -np.log(density + eps)

plt.figure(figsize=(6,4))
plt.plot(bin_centers, S_overall, marker='o')
plt.xlabel("a")
plt.ylabel("S(a)")
plt.title("RSVP entropy field S(a) from stationary distribution")
plt.grid(True)
plt.show()


In [ ]:

class ValueNet(nn.Module):
    \"\"\"Value potential network: (a,z) -> Phi(a,z)\"\"\"
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )
    def forward(self, a, z):
        x = torch.cat([a, z], dim=1)
        return self.net(x)

value_net = ValueNet().to(device)
value_net


In [ ]:

def rollout_value(a0, z0_idx, policy_net, r, w, H=50):
    \"\"\"Monte Carlo finite-horizon approximation of value starting from (a0,z0).\"\"\"
    a = float(a0)
    z_idx = int(z0_idx)
    total = 0.0
    disc = 1.0
    for t in range(H):
        z_val = z_states[z_idx]
        y = w * z_val + (1 + r) * a
        # policy
        a_tensor = torch.tensor([[a]], dtype=torch.float32, device=device)
        z_tensor = torch.tensor([[z_val]], dtype=torch.float32, device=device)
        with torch.no_grad():
            a_next = policy_net(a_tensor, z_tensor).cpu().numpy().item()
        c = y - a_next
        c = max(c, 1e-8)
        u = (c**(1 - sigma)) / (1 - sigma) if sigma != 1 else np.log(c)
        total += disc * u
        disc *= beta
        a = a_next
        z_idx = sample_next_z(np.array([z_idx]))[0]
    return total

# Build training data for value_net from stationary samples
N_val_samples = 4000
idx = np.random.choice(len(samples_a), size=N_val_samples, replace=False)
a_samples_val = samples_a[idx]
z_samples_val_idx = samples_z[idx]
z_samples_val = z_states[z_samples_val_idx]

vals = []
for a0, z0_idx in zip(a_samples_val, z_samples_val_idx):
    vals.append(rollout_value(a0, z0_idx, policy_net, r_eq, w_eq, H=40))
vals = np.array(vals, dtype=np.float32)

a_val_t = torch.tensor(a_samples_val, dtype=torch.float32, device=device).view(-1,1)
z_val_t = torch.tensor(z_samples_val, dtype=torch.float32, device=device).view(-1,1)
y_val_t = torch.tensor(vals, dtype=torch.float32, device=device).view(-1,1)

optimizer_val = optim.Adam(value_net.parameters(), lr=1e-3)
val_losses = []
for epoch in range(200):
    optimizer_val.zero_grad()
    pred = value_net(a_val_t, z_val_t)
    loss_val = ((pred - y_val_t)**2).mean()
    loss_val.backward()
    optimizer_val.step()
    val_losses.append(loss_val.item())
    if (epoch+1) % 50 == 0:
        print(f"[value_net] epoch {epoch+1:3d} | MSE={loss_val.item():.5f}")


In [ ]:

plt.figure(figsize=(6,4))
plt.plot(val_losses)
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("ValueNet training loss")
plt.grid(True)
plt.show()


In [ ]:

a_grid = np.linspace(a_min, a_max, 100, dtype=np.float32)
a_t = torch.tensor(a_grid, dtype=torch.float32, device=device).view(-1,1)

plt.figure(figsize=(6,4))
with torch.no_grad():
    for z_val in z_states:
        z_t = torch.full_like(a_t, z_val)
        phi_t = value_net(a_t, z_t)
        phi = phi_t.cpu().numpy().reshape(-1)
        plt.plot(a_grid, phi, label=f"z={z_val:.1f}")

plt.xlabel("a")
plt.ylabel("Phi(a,z)")
plt.title("Approximate value potential field Φ(a,z)")
plt.legend()
plt.grid(True)
plt.show()
